In [ ]:
import pandas as pd
import numpy as np

In [ ]:
!python -c "import numpy; print(numpy.__version__)"

In [ ]:
# Select your NDD
ndd = 'DEM'

In [ ]:
#Load controls created in step 02
controls = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/controls_60_n162460.csv')
controls[f'{ndd}_DATE'] = np.nan
controls = controls.rename(columns = {'person_id':'ID'}) 
controls = controls[['ID', 'date_of_birth', 'sex_at_birth', f'{ndd}_DATE']]
controls

In [ ]:
# Combine cases and controls
df = pd.concat([controls])

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df = df.sort_values(by = f'{ndd}_DATE')
df = df.drop_duplicates(subset = 'ID', keep = 'first')

#Check to make sure no duplicate IDs
print(df.ID.value_counts())

df

In [ ]:
import pandas
import os

# This pulls people who have died and have a recorded death date
cdr = os.environ["WORKSPACE_CDR"]

dataset_death_sql = f"""
SELECT DISTINCT person_id, death_date
FROM `{cdr}.death`


"""

dataset_death_person_df = pandas.read_gbq(
    dataset_death_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_death_person_df

In [ ]:
dataset_death_person_df.to_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/death_date.csv', header = True, index = False)

In [ ]:
# Add DEATH YEAR from file created above
d = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/death_date.csv')
d = d.rename(columns = {'person_id':'ID', 'death_date':'DATE_OF_DEATH'})
d

In [ ]:
#Merge with cases/controls
df = controls.merge(d, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
# eliminate people who were diagnosed at death
df = df[df[f'{ndd}_DATE'] != df['DATE_OF_DEATH']]
df

In [ ]:
import pandas as pd
DATASET = %env WORKSPACE_CDR
print(DATASET)

In [ ]:
# This code pulls the primary consent date for AoU
import pandas as pd
DATASET = %env WORKSPACE_CDR
consent = pd.read_gbq(f'''
SELECT DISTINCT person_id, MIN(observation_date) AS primary_consent_date
FROM `{DATASET}.concept`
JOIN `{DATASET}.concept_ancestor` on concept_id = ancestor_concept_id
JOIN `{DATASET}.observation` on descendant_concept_id = observation_source_concept_id
WHERE concept_name = 'Consent PII' AND concept_class_id = 'Module'
GROUP BY 1''')
consent.head()

In [ ]:
consent.to_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/primary_consent.csv', header = True, index = False)

In [ ]:
# Add recruit year from file created in step 02
r = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/other/primary_consent.csv')
r = r.rename(columns = {'person_id':'ID', 'primary_consent_date':'recruit_date'})
r

In [ ]:
#Merge with cases/controls
df = df.merge(r, left_on = 'ID', right_on = 'ID', how = 'left')
df

In [ ]:
controls = df
controls

# Add calculated fields

In [ ]:
STUDY_ENDS = '2024-01-01'

#For people without NDD, break into dead and alive
alive = controls[controls['DATE_OF_DEATH'].isna()]
dead = controls[~controls['DATE_OF_DEATH'].isna()]

#Calculate the tenure for people who are still alive, i.e. the time from the beginning of the study to the end of study
alive['tenure'] = (pd.to_datetime(STUDY_ENDS) - pd.to_datetime(alive['recruit_date'])).dt.days/365

#Add age_at_tenure for people who are still alive
alive['age_at_tenure'] = (pd.to_datetime(STUDY_ENDS) - pd.to_datetime(alive['date_of_birth'])).dt.days/365

#Calculate the tenure for people who are still dead, i.e. the time from the beginning of the study to the end of study
dead['tenure'] = (pd.to_datetime(dead['DATE_OF_DEATH']) - pd.to_datetime(dead['recruit_date'])).dt.days/365

#Add age_at_tenure for people who are dead
dead['age_at_tenure'] = (pd.to_datetime(dead['DATE_OF_DEATH']) - pd.to_datetime(dead['date_of_birth'])).dt.days/365

#Combine two groups
controls = pd.concat([alive, dead])
print(len(controls))
controls

In [ ]:
#Remove cases without TOWNEND 
#controls = controls[~controls['TOWNSEND'].isna()]
controls = controls[~controls['sex_at_birth'].isna()]
controls = controls[~controls['date_of_birth'].isna()]
#controls = controls[~controls[f'{ndd}_DATE'].isna()]
controls

In [ ]:
controls['ID'] = controls['ID'].astype("Int64")
controls = controls.sort_values(by = 'ID')
controls

In [ ]:
controls.tenure.value_counts()

In [ ]:
#START_DATE = '1999-01-01'
START_DATE = '2015-01-01'
STUDY_ENDS = '2024-01-01'

In [ ]:
# Create new tenure length 

#drop old tenure column
controls = controls.drop(columns = 'tenure')

#For people without NDD, break into dead and alive
alive = controls[controls['DATE_OF_DEATH'].isna()]
dead = controls[~controls['DATE_OF_DEATH'].isna()]

#Calculate the tenure for people who are still alive, i.e. the time from the beginning of the study to the end of study
#alive['tenure'] = (pd.to_datetime(STUDY_ENDS) - pd.to_datetime(alive['start_date'])).dt.days/365
alive['tenure'] = (pd.to_datetime(STUDY_ENDS) - pd.to_datetime(START_DATE)).days / 365

#Add age_at_tenure for people who are still alive
alive['age_at_tenure'] = (pd.to_datetime(STUDY_ENDS) - pd.to_datetime(alive['date_of_birth'])).dt.days/365
alive['tenure_date'] = pd.to_datetime(STUDY_ENDS)

#Calculate the tenure for people who are still dead, i.e. the time from the beginning of the study to the end of study
#dead['tenure'] = (pd.to_datetime(dead['DATE_OF_DEATH']) - pd.to_datetime(dead['start_date'])).dt.days/365
dead['tenure'] = (pd.to_datetime(dead['DATE_OF_DEATH']) - pd.to_datetime(START_DATE)).dt.days/365


#Add age_at_tenure for people who are dead
dead['age_at_tenure'] = (pd.to_datetime(dead['DATE_OF_DEATH']) - pd.to_datetime(dead['date_of_birth'])).dt.days/365
dead['tenure_date'] = dead['DATE_OF_DEATH']

#Combine two groups
controls = pd.concat([alive, dead])
print(len(controls))
controls

In [ ]:
# removed early onset
controls = controls[controls['age_at_tenure'] >= 60]
controls

In [ ]:
print(len(controls))
controls = controls[controls['tenure'] >= 5]
print(len(controls))
controls

In [ ]:
controls.sex_at_birth.value_counts()

In [ ]:
controls.to_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/CONTROLS_with_tenure_JUNE_25_2026.csv', header = True, index = None)